# Практична робота №3. Приведення до tidy data

**Студент:** Панасюк Дмитро  
**Номер за списком групи:** 15  
**Варіант:** №5 — **Дніпро**  
**Мета:** перетворення wide ↔ tidy (long), класифікація шкал вимірювання, читання та запис CSV і JSON.

Використано **навчальні значення середньомісячної хмарності (%)** саме для Дніпра з таблиці завдання.

## Підготовка середовища

Імпортуємо pandas. Ноутбук запускається в Google Colab або локальному Jupyter із встановленою бібліотекою `pandas`.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

print("Версія pandas:", pd.__version__)

Версія pandas: 2.2.3


## Завдання 1. Побудова wide-таблиці

Один рядок відповідає місту **Дніпро**, а окремі 12 стовпців — місяцям року. Для місячних вимірювань це **wide-формат**.

In [2]:
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

wide = pd.DataFrame({
    "місто": ["Дніпро"],
    "Січ": [74],
    "Лют": [70],
    "Бер": [63],
    "Кві": [53],
    "Тра": [45],
    "Чер": [39],
    "Лип": [35],
    "Сер": [37],
    "Вер": [47],
    "Жов": [60],
    "Лис": [72],
    "Гру": [78],
})

print("Форма wide-таблиці:", wide.shape)
display(wide)
assert wide.shape == (1, 13)
assert wide.loc[0, months].tolist() == [74, 70, 63, 53, 45, 39, 35, 37, 47, 60, 72, 78]

Форма wide-таблиці: (1, 13)


,місто,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру
0,Дніпро,74,70,63,53,45,39,35,37,47,60,72,78


## Завдання 2. Приведення до tidy (`melt`)

Назви місяців із 12 окремих стовпців перетворюємо на значення **одного** стовпця `місяць`, а показники — на значення стовпця `хмарність`. У кожному рядку буде одне спостереження: хмарність у конкретному місті за конкретний місяць.

In [3]:
tidy = wide.melt(
    id_vars="місто",
    value_vars=months,
    var_name="місяць",
    value_name="хмарність",
)

print("Форма tidy-таблиці:", tidy.shape)
display(tidy)
assert tidy.shape == (12, 3)
assert tidy.columns.tolist() == ["місто", "місяць", "хмарність"]

Форма tidy-таблиці: (12, 3)


,місто,місяць,хмарність
0,Дніпро,Січ,74
1,Дніпро,Лют,70
2,Дніпро,Бер,63
3,Дніпро,Кві,53
4,Дніпро,Тра,45
5,Дніпро,Чер,39
6,Дніпро,Лип,35
7,Дніпро,Сер,37
8,Дніпро,Вер,47
9,Дніпро,Жов,60


**Пояснення:** у wide-таблиці є 1 рядок і 13 стовпців (місто + 12 місяців). Після `melt()` кожен із 12 місячних показників стає окремим рядком, тому отримуємо **12 рядків × 3 стовпці**. Кількість вимірювань не змінилася — змінилося лише їх представлення.

## Завдання 3. Класифікація змінних за шкалою вимірювання

| Стовпець | Шкала | Пояснення |
|---|---|---|
| **місто** | **Номінальна** | Назва міста є категорією без природного ранжування або числової відстані між значеннями. |
| **місяць** | **Порядкова** | Місяці мають природну послідовність у межах року (січень, лютий, …, грудень), хоч це циклічна календарна категорія. |
| **хмарність** | **Відносна (шкала відношень)** | Це числова частка неба, вкрита хмарами, у відсотках: 0% означає відсутність хмарності, а співвідношення значень має зміст. |

## Завдання 4. Зворотне перетворення (`pivot`) і перевірка

Повертаємо **tidy → wide** через `pivot()`. Після цього впорядковуємо стовпці за календарем і **програмно звіряємо** таблицю з початковою.

In [4]:
pivoted = tidy.pivot(index="місто", columns="місяць", values="хмарність")
print("Результат pivot() (порядок місяців може відрізнятися):")
display(pivoted)

wide_restored = (
    pivoted.reindex(columns=months)
    .reset_index()
    .rename_axis(columns=None)
)
print("Відновлена wide-таблиця в порядку січень–грудень:")
display(wide_restored)

pd.testing.assert_frame_equal(wide, wide_restored)
print("Перевірку пройдено: усі значення та стовпці початкової і відновленої таблиць збігаються.")

Результат pivot() (порядок місяців може відрізнятися):


місяць,Бер,Вер,Гру,Жов,Кві,Лип,Лис,Лют,Сер,Січ,Тра,Чер
місто,,,,,,,,,,,,
Дніпро,63,47,78,60,53,35,72,70,37,74,45,39


Відновлена wide-таблиця в порядку січень–грудень:


,місто,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру
0,Дніпро,74,70,63,53,45,39,35,37,47,60,72,78


Перевірку пройдено: усі значення та стовпці початкової і відновленої таблиць збігаються.


**Пояснення:** `pivot()` повертає місяці зі значень рядків у назви стовпців. За замовчуванням стовпці можуть іти за алфавітом, тому ми явно відновили календарний порядок перед порівнянням. Значення хмарності **не змінилися**.

## Завдання 5. CSV і JSON: запис, читання, порівняння

Зберігаємо **tidy-таблицю** у `variant.csv` **без індексу** та `variant.json` у форматі **`orient="records"`**. Файли записуються в поточну робочу папку ноутбука.

In [5]:
tidy.to_csv("variant.csv", index=False, encoding="utf-8")
tidy.to_json("variant.json", orient="records", force_ascii=False, indent=2)

reloaded_csv = pd.read_csv("variant.csv", encoding="utf-8")
reloaded_json = pd.read_json("variant.json", orient="records")

pd.testing.assert_frame_equal(tidy, reloaded_csv)
pd.testing.assert_frame_equal(tidy, reloaded_json)

print("Форма після читання CSV:", reloaded_csv.shape)
print("Форма після читання JSON:", reloaded_json.shape)
print("Результат: дані, порядок рядків, назви стовпців і типи збігаються з tidy-таблицею.")
print("Створені файли:", Path("variant.csv").resolve(), "і", Path("variant.json").resolve(), sep="\n")

Форма після читання CSV: (12, 3)
Форма після читання JSON: (12, 3)
Результат: дані, порядок рядків, назви стовпців і типи збігаються з tidy-таблицею.
Створені файли:
/mnt/data/variant.csv
і
/mnt/data/variant.json


In [6]:
print("===== ЗМІСТ variant.csv =====")
print(Path("variant.csv").read_text(encoding="utf-8"))
print("===== ЗМІСТ variant.json =====")
print(Path("variant.json").read_text(encoding="utf-8"))

===== ЗМІСТ variant.csv =====
місто,місяць,хмарність
Дніпро,Січ,74
Дніпро,Лют,70
Дніпро,Бер,63
Дніпро,Кві,53
Дніпро,Тра,45
Дніпро,Чер,39
Дніпро,Лип,35
Дніпро,Сер,37
Дніпро,Вер,47
Дніпро,Жов,60
Дніпро,Лис,72
Дніпро,Гру,78

===== ЗМІСТ variant.json =====
[
  {
    "місто":"Дніпро",
    "місяць":"Січ",
    "хмарність":74
  },
  {
    "місто":"Дніпро",
    "місяць":"Лют",
    "хмарність":70
  },
  {
    "місто":"Дніпро",
    "місяць":"Бер",
    "хмарність":63
  },
  {
    "місто":"Дніпро",
    "місяць":"Кві",
    "хмарність":53
  },
  {
    "місто":"Дніпро",
    "місяць":"Тра",
    "хмарність":45
  },
  {
    "місто":"Дніпро",
    "місяць":"Чер",
    "хмарність":39
  },
  {
    "місто":"Дніпро",
    "місяць":"Лип",
    "хмарність":35
  },
  {
    "місто":"Дніпро",
    "місяць":"Сер",
    "хмарність":37
  },
  {
    "місто":"Дніпро",
    "місяць":"Вер",
    "хмарність":47
  },
  {
    "місто":"Дніпро",
    "місяць":"Жов",
    "хмарність":60
  },
  {
    "місто":"Дніпро",
    "місяць":"Лис",

**Порівняння форматів:** у **CSV** перший рядок містить назви стовпців, а кожен наступний — значення одного спостереження, розділені комами (наприклад, `Дніпро,Січ,74`). У **JSON** із `orient="records"` таблиця зберігається як **масив об'єктів**: кожен об'єкт має іменовані поля `місто`, `місяць`, `хмарність`. У CSV немає явних типів даних і вкладених структур, а JSON зберігає числа як числа та підтримує складніші структури.

## Письмові відповіді на контрольні запитання

**1. Чому `місто` — номінальна шкала, а `хмарність` — шкала відношень?**  
`місто` — це лише назва категорії: міста не мають природного числового порядку чи арифметичного змісту. `хмарність` — вимірювана кількісна частка у відсотках із природним нулем (0% — немає хмарності), тому можна порівнювати не лише різницю, а й відношення показників (наприклад, 60% удвічі більше за 30%).

**2. Чому wide-таблиця з місяцями в стовпцях не є tidy і що змінює `melt()`?**  
За принципами tidy data кожна змінна повинна мати власний стовпець, а кожне спостереження — власний рядок. У вихідній таблиці значення однієї змінної — **місяць** — закодовані назвами 12 стовпців, тому цей принцип порушено. `melt()` створює один стовпець `місяць` і один стовпець `хмарність`: кожен місяць стає окремим спостереженням, а місто повторюється в усіх 12 рядках.

**3. Чим принципово відрізняються CSV і JSON для тієї самої таблиці?**  
**CSV** — це рядки тексту з роздільниками: заголовок задає імена колонок, а кожен наступний рядок — значення без явно записаних типів. **JSON** у форматі `records` — це масив об'єктів із парами «ключ–значення»: кожен запис містить назви полів, числа залишаються числовими значеннями. JSON також підтримує вкладені об'єкти й масиви, на відміну від звичайного табличного CSV.

---

**Висновок:** для навчальних даних Дніпра побудовано wide-таблицю, отримано tidy-формат за допомогою `melt()`, визначено шкали вимірювання трьох змінних, виконано зворотне перетворення `pivot()` і підтверджено повний збіг даних. Таблицю записано у CSV і JSON та перевірено її зчитування без втрат.